In [0]:
%pip install pypdf sentence-transformers faiss-cpu groq

In [0]:
dbutils.library.restartPython()

In [0]:
pdf_path = "/Volumes/retail_project/default/rag_docs/AI-reseach-paper.pdf"

from pypdf import PdfReader

def read_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text

text = read_pdf(pdf_path)

print(text[:1000])

In [0]:
def create_chunks(text,
                  chunk_size=500,
                  overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks


chunks = create_chunks(text)

print("Total Chunks:", len(chunks))
print(chunks[0])

In [0]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True
)

print(embeddings.shape)

In [0]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension) # to retrieve back from chunks
index.add(embeddings)

print(index.ntotal)

In [0]:
faiss_path = "/Volumes/retail_project/default/rag_docs/pdf_index.faiss"

faiss.write_index(index, faiss_path)

In [0]:
import pickle

chunk_path = "/Volumes/retail_project/default/rag_docs/chunks.pkl"

with open(chunk_path, "wb") as f:
    pickle.dump(chunks, f)

In [0]:
import faiss
import pickle

index = faiss.read_index(
    "/Volumes/retail_project/default/rag_docs/pdf_index.faiss"
)

with open(
    "/Volumes/retail_project/default/rag_docs/chunks.pkl",
    "rb"
) as f:

    chunks = pickle.load(f)

In [0]:
query = "what is applications of AI"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
)

In [0]:
print(query_embedding.shape)

In [0]:
scores, indices = index.search(
    query_embedding,
    k=3
)

print(scores)
print(indices)

In [0]:
retrieved_chunks = [chunks[idx] for idx in indices[0]]

context = "\n\n".join(retrieved_chunks)

print(context)

In [0]:
from getpass import getpass

groq_api_key = getpass("Enter api key")

In [0]:
from groq import Groq

client = Groq(api_key=groq_api_key)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "system",
            "content": (
                "You are a helpful PDF question-answering assistant. "
                "Answer only from the supplied context. "
                "If the answer is not present in the context, say: "
                "'I could not find the answer in the provided PDF.'"
            )
        },
        {
            "role": "user",
            "content": f"""
Context from the PDF:
{context}

Question:
{query}
"""
        }
    ],
    temperature=0.1,
    max_completion_tokens=500
)

answer = response.choices[0].message.content

print("Answer:")
print(answer)